# Output Visualization Notebook

This notebook reads already-generated CSV files from `outputs/` and creates comparison tables and plots. It does not call Neuronpedia.

It supports the current competence output files and older French `_fr.csv` expertise-style files by normalizing them into one shared table.


## 1. Setup


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


if Path.cwd().name == "profession_concept_overlap":
    BASE_DIR = Path.cwd()
else:
    BASE_DIR = Path("profession_concept_overlap")

OUTPUT_DIR = BASE_DIR / "outputs"
EPSILON = 1e-9

GENDER_COLORS = {"control": "#b8b8b8", "male": "#8ecae6", "female": "#f4a6c1"}
LANGUAGE_ORDER = ["english", "traditional_chinese", "french"]
GENDER_ORDER = ["control", "male", "female"]

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 200)

OUTPUT_DIR


## 2. Load Output Files


In [ ]:
def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if path.exists() and path.stat().st_size > 0:
        return pd.read_csv(path)
    return pd.DataFrame()


def add_missing_language(frame: pd.DataFrame, language: str) -> pd.DataFrame:
    if frame.empty:
        return frame
    frame = frame.copy()
    if "language" not in frame.columns:
        frame["language"] = language
    elif frame["language"].isna().any():
        frame["language"] = frame["language"].fillna(language)
    return frame


trait_records = read_csv_if_exists(OUTPUT_DIR / "profession_gender_trait_fixed_feature_records.csv")
trait_group_summary = read_csv_if_exists(OUTPUT_DIR / "profession_gender_trait_group_summary.csv")
trait_feature_summary = read_csv_if_exists(OUTPUT_DIR / "profession_gender_trait_feature_summary_long.csv")

french_records = add_missing_language(
    read_csv_if_exists(OUTPUT_DIR / "profession_gender_expertise_fixed_feature_records_fr.csv"),
    "french",
)
french_group_summary = add_missing_language(
    read_csv_if_exists(OUTPUT_DIR / "profession_gender_expertise_group_summary_fr.csv"),
    "french",
)
french_feature_summary = add_missing_language(
    read_csv_if_exists(OUTPUT_DIR / "profession_gender_expertise_feature_summary_long_fr.csv"),
    "french",
)

for frame in [trait_records, trait_group_summary, trait_feature_summary, french_records, french_group_summary, french_feature_summary]:
    if not frame.empty and "trait_category" not in frame.columns:
        frame["trait_category"] = "competence"

print("trait_records", trait_records.shape)
print("trait_group_summary", trait_group_summary.shape)
print("french_records", french_records.shape)
print("french_group_summary", french_group_summary.shape)


## 3. Normalize And Combine


In [ ]:
def normalize_records(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()
    needed = [
        "language",
        "trait_category",
        "profession",
        "gender_group",
        "prompt",
        "prompt_pair",
        "feature",
        "source",
        "feature_index",
        "attribute_activation",
        "content_activation",
        "role_activation",
        "max_activation",
    ]
    normalized = frame.copy()
    for column in needed:
        if column not in normalized.columns:
            normalized[column] = pd.NA
    return normalized[needed]


def normalize_group_summary(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()
    needed = [
        "language",
        "trait_category",
        "profession",
        "gender_group",
        "prompt_count",
        "tested_feature_count",
        "mean_attribute_activation",
        "mean_content_activation",
        "mean_role_activation",
    ]
    normalized = frame.copy()
    for column in needed:
        if column not in normalized.columns:
            normalized[column] = pd.NA
    return normalized[needed]


activation_records = pd.concat(
    [normalize_records(trait_records), normalize_records(french_records)],
    ignore_index=True,
).dropna(subset=["language", "profession", "gender_group"], how="any")

group_summary = pd.concat(
    [normalize_group_summary(trait_group_summary), normalize_group_summary(french_group_summary)],
    ignore_index=True,
).dropna(subset=["language", "profession", "gender_group"], how="any")

# If a summary CSV is missing, derive group_summary from raw activation records.
if group_summary.empty and not activation_records.empty:
    group_summary = (
        activation_records
        .groupby(["language", "trait_category", "profession", "gender_group"], as_index=False)
        .agg(
            prompt_count=("prompt", "nunique"),
            tested_feature_count=("feature", "nunique"),
            mean_attribute_activation=("attribute_activation", "mean"),
            mean_content_activation=("content_activation", "mean"),
            mean_role_activation=("role_activation", "mean"),
        )
    )

for column in ["mean_attribute_activation", "mean_content_activation", "mean_role_activation"]:
    if column in group_summary.columns:
        group_summary[column] = pd.to_numeric(group_summary[column], errors="coerce")

group_summary["language"] = pd.Categorical(group_summary["language"], categories=LANGUAGE_ORDER, ordered=True)
group_summary["gender_group"] = pd.Categorical(group_summary["gender_group"], categories=GENDER_ORDER, ordered=True)
group_summary = group_summary.sort_values(["language", "profession", "gender_group"]).reset_index(drop=True)

display(group_summary.head(20))
print("languages:", list(group_summary["language"].dropna().unique()))
print("professions:", sorted(group_summary["profession"].dropna().unique()))


## 4. Difference Tables


In [ ]:
def pct_relative_to_control(value: pd.Series, control: pd.Series) -> pd.Series:
    denominator = control.where(control.abs() > EPSILON)
    return ((value - control) / denominator) * 100


def signed_pct_difference(left: pd.Series, right: pd.Series) -> pd.Series:
    denominator = ((left.abs() + right.abs()) / 2).where(lambda s: s > EPSILON)
    return ((left - right) / denominator) * 100


group_difference_table = (
    group_summary
    .pivot_table(
        index=["language", "trait_category", "profession"],
        columns="gender_group",
        values="mean_attribute_activation",
        fill_value=0.0,
        observed=False,
    )
    .reset_index()
)

for column in ["control", "male", "female"]:
    if column not in group_difference_table.columns:
        group_difference_table[column] = 0.0

group_difference_table["male_minus_control"] = group_difference_table["male"] - group_difference_table["control"]
group_difference_table["female_minus_control"] = group_difference_table["female"] - group_difference_table["control"]
group_difference_table["male_minus_female"] = group_difference_table["male"] - group_difference_table["female"]
group_difference_table["male_pct_vs_control"] = pct_relative_to_control(group_difference_table["male"], group_difference_table["control"])
group_difference_table["female_pct_vs_control"] = pct_relative_to_control(group_difference_table["female"], group_difference_table["control"])
group_difference_table["male_female_pct_gap"] = signed_pct_difference(group_difference_table["male"], group_difference_table["female"])

male_female_comparison_table = group_difference_table[
    ["language", "profession", "male", "female", "male_minus_female", "male_female_pct_gap"]
].copy()
male_female_comparison_table["higher_group"] = "tie"
male_female_comparison_table.loc[male_female_comparison_table["male_female_pct_gap"] > 0, "higher_group"] = "male"
male_female_comparison_table.loc[male_female_comparison_table["male_female_pct_gap"] < 0, "higher_group"] = "female"
male_female_comparison_table = male_female_comparison_table[
    ["language", "profession", "higher_group", "male", "female", "male_minus_female", "male_female_pct_gap"]
].sort_values(["language", "profession"])

display(group_difference_table)
display(male_female_comparison_table)


## 5. Styled Male-Female Comparison Table


In [ ]:
def format_signed_pct(value: float) -> str:
    if pd.isna(value):
        return ""
    return f"{value:+.1f}%"


def blend_rgb(start: tuple[int, int, int], end: tuple[int, int, int], weight: float) -> str:
    weight = max(0.0, min(1.0, float(weight)))
    rgb = [round(start_value + (end_value - start_value) * weight) for start_value, end_value in zip(start, end)]
    return f"rgb({rgb[0]}, {rgb[1]}, {rgb[2]})"


def style_signed_percentage_table(frame: pd.DataFrame, pct_columns: list[str]):
    max_abs = frame[pct_columns].abs().max().max()
    max_abs = float(max_abs) if pd.notna(max_abs) and max_abs > EPSILON else 1.0
    white = (255, 255, 255)
    male_blue = (142, 202, 230)
    female_pink = (244, 166, 193)

    def color_signed_pct(value):
        if pd.isna(value):
            return ""
        weight = min(abs(float(value)) / max_abs, 1.0)
        if value > 0:
            background = blend_rgb(white, male_blue, weight)
        elif value < 0:
            background = blend_rgb(white, female_pink, weight)
        else:
            background = "white"
        return f"background-color: {background}; color: #111111; font-weight: 600"

    return (
        frame.style
        .format({column: format_signed_pct for column in pct_columns})
        .format({"male": "{:.3f}", "female": "{:.3f}", "male_minus_female": "{:+.3f}"})
        .map(color_signed_pct, subset=pct_columns)
        .set_properties(subset=pct_columns, **{"text-align": "center"})
    )


display(style_signed_percentage_table(male_female_comparison_table, ["male_female_pct_gap"]))


## 6. Mean Activation Bar Plots


In [ ]:
def colors_for_columns(columns: list[str]) -> list[str]:
    return [GENDER_COLORS.get(column, "#b8b8b8") for column in columns]


for language, language_frame in group_summary.groupby("language", observed=True):
    if language_frame.empty:
        continue
    plot_frame = language_frame.pivot_table(
        index="profession",
        columns="gender_group",
        values="mean_attribute_activation",
        fill_value=0.0,
        observed=False,
    )
    plot_frame = plot_frame[[column for column in GENDER_ORDER if column in plot_frame.columns]]
    ax = plot_frame.plot.bar(
        figsize=(9, 4),
        color=colors_for_columns(list(plot_frame.columns)),
        title=f"{language}: mean competence activation by profession",
        width=0.78,
    )
    ax.set_xlabel("")
    ax.set_ylabel("mean attribute-token activation")
    ax.legend(title="")
    plt.xticks(rotation=0)
    plt.show()


## 7. Signed Male-Female Gap Plot


In [ ]:
plot_frame = male_female_comparison_table.copy()
plot_frame["label"] = plot_frame["language"].astype(str) + " | " + plot_frame["profession"].astype(str)
plot_frame = plot_frame.sort_values("male_female_pct_gap")
colors = [GENDER_COLORS["male"] if value >= 0 else GENDER_COLORS["female"] for value in plot_frame["male_female_pct_gap"]]

fig, ax = plt.subplots(figsize=(9, max(4, 0.45 * len(plot_frame))), constrained_layout=True)
ax.barh(plot_frame["label"], plot_frame["male_female_pct_gap"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Male vs female competence activation gap")
ax.set_xlabel("signed % gap: positive = male higher, negative = female higher")
ax.set_ylabel("")
plt.show()


## 8. Control-Relative Table


In [ ]:
control_relative_table = group_difference_table[
    [
        "language",
        "profession",
        "control",
        "male",
        "female",
        "male_pct_vs_control",
        "female_pct_vs_control",
    ]
].sort_values(["language", "profession"])

display(style_signed_percentage_table(control_relative_table, ["male_pct_vs_control", "female_pct_vs_control"]))
